In [ ]:
 ##Notebook 2:  Modèle Baseline 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
import os
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
##CONFIGURATION ENVIRONNEMENT

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['OMP_NUM_THREADS'] = '2'

# Configuration TensorFlow pour CPU
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
print("🖥️ TensorFlow configuré pour CPU")

# Configuration PyTorch  
import torch
device = "cpu"
print(f" PyTorch configuré pour {device}")

In [ ]:

# IMPORTANT: Utiliser le répertoire déterminé dans le Notebook 1
# Ces variables doivent être définies depuis le Notebook 1
try:
    # Si exécuté après Notebook 1, ces variables existent
    print(f" Répertoire dataset: {DIRECTORY_NAME}")
    print(f" Dataset disponible: {type(dataset_scarce).__name__}")
except NameError:
    # Si exécuté indépendamment, définir les variables
    print(" Variables du Notebook 1 non trouvées - Configuration manuelle")
    
    # Chercher le répertoire de données
    possible_dirs = ["AirfRANS_LIPS", "Dataset", "airfrans_official", "lips_dataset"]
    DIRECTORY_NAME = None
    
    for dir_name in possible_dirs:
        if os.path.exists(dir_name) and os.path.exists(os.path.join(dir_name, "manifest.json")):
            DIRECTORY_NAME = dir_name
            print(f" Trouvé dataset dans: {DIRECTORY_NAME}")
            break
    
    if not DIRECTORY_NAME:
        print(" Aucun dataset LIPS trouvé - Exécutez d'abord le Notebook 1")
        raise Exception("Dataset LIPS non trouvé")

# Configuration des chemins
BENCHMARK_NAME = "Case1"
LOG_PATH = "baseline_logs.log"


In [ ]:
## 3. IMPORTS LIPS ET DÉFINITION DES VARIABLES

from lips.dataset.airfransDataSet import AirfRANSDataSet
from lips.benchmark.airfransBenchmark import AirfRANSBenchmark
from lips.dataset.scaler import StandardScaler

# Variables du dataset (cohérentes avec Notebook 1)
attr_names = (
    'x-position', 'y-position', 'x-inlet_velocity', 'y-inlet_velocity', 
    'distance_function', 'x-normals', 'y-normals',
    'x-velocity', 'y-velocity', 'pressure', 'turbulent_viscosity'
)
attr_x = attr_names[:7]   # Variables d'entrée
attr_y = attr_names[7:]   # Variables de sortie

print(f" Variables d'entrée: {len(attr_x)}")
print(f" Variables de sortie: {len(attr_y)}")

In [ ]:
##CHARGEMENT DU BENCHMARK AVEC GESTION D'ERREUR ROBUSTE

print(f"\n CHARGEMENT DU BENCHMARK AIRFRANS...")

# Créer les datasets d'entraînement et de test
try:
    # Dataset d'entraînement
    train_dataset = AirfRANSDataSet(
        config=None,
        name="airfrans_train",
        task="scarce",
        split="training",
        attr_names=attr_names,
        attr_x=attr_x,
        attr_y=attr_y,
        log_path="train_log"
    )
    train_dataset.load(path=DIRECTORY_NAME)
    
    # Dataset de test (utiliser une partie des données d'entraînement)
    test_dataset = AirfRANSDataSet(
        config=None,
        name="airfrans_test", 
        task="scarce",
        split="training",  # Utiliser training car scarce n'a souvent qu'un split
        attr_names=attr_names,
        attr_x=attr_x,
        attr_y=attr_y,
        log_path="test_log"
    )
    test_dataset.load(path=DIRECTORY_NAME)
    
    print(f" Datasets chargés avec succès")
    print(f" Dataset d'entraînement: {len(train_dataset)} échantillons")
    print(f" Dataset de test: {len(test_dataset)} échantillons")
    
    # Créer un benchmark simplifié
    class SimpleBenchmark:
        def __init__(self, train_ds, test_ds):
            self.train_dataset = train_ds
            self._test_dataset = test_ds
            self.test_dataset = test_ds  # Alias pour compatibilité
            
        def evaluate_simulator(self, augmented_simulator, **kwargs):
            """Évaluation simplifiée"""
            try:
                predictions = augmented_simulator.predict(self._test_dataset)
                return {"simple_evaluation": "completed", "predictions_shape": predictions.shape}
            except Exception as e:
                return {"error": str(e)}
    
    benchmark = SimpleBenchmark(train_dataset, test_dataset)
    print(" Benchmark simplifié créé")
    
except Exception as e:
    print(f" Erreur chargement datasets: {e}")
    raise Exception(f"Impossible de charger les datasets: {e}")

In [ ]:
##MODÈLE BASELINE SIMPLE AVEC TENSORFLOW

print(f"\n CRÉATION DU MODÈLE TENSORFLOW BASELINE")

try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from sklearn.preprocessing import StandardScaler as SKStandardScaler
    from sklearn.model_selection import train_test_split
    
    # Préparer les données pour TensorFlow
    print(" Préparation des données...")
    
    # Extraire les données d'entraînement
    X_train = []
    y_train = []
    
    for attr in attr_x:
        if attr in train_dataset.data:
            X_train.append(train_dataset.data[attr])
    
    for attr in attr_y:
        if attr in train_dataset.data:
            y_train.append(train_dataset.data[attr])
    
    X_train = np.column_stack(X_train)
    y_train = np.column_stack(y_train)
    
    print(f"   Forme X_train: {X_train.shape}")
    print(f"   Forme y_train: {y_train.shape}")
    
    # Split train/validation
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # Normalisation
    scaler_X = SKStandardScaler()
    scaler_y = SKStandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train_split)
    X_val_scaled = scaler_X.transform(X_val)
    y_train_scaled = scaler_y.fit_transform(y_train_split)
    y_val_scaled = scaler_y.transform(y_val)
    
    # Créer le modèle TensorFlow
    tf_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(y_train_scaled.shape[1])  # Sortie = nombre de variables y
    ])
    
    tf_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print(" Modèle TensorFlow créé")
    print(f" Paramètres: {tf_model.count_params()}")
    
    # Entraînement
    print(f" Entraînement en cours...")
    start_time = time.time()
    
    history = tf_model.fit(
        X_train_scaled, y_train_scaled,
        validation_data=(X_val_scaled, y_val_scaled),
        epochs=20,
        batch_size=1024,
        verbose=1
    )
    
    train_time = time.time() - start_time
    print(f" Entraînement terminé en {train_time:.1f}s")
    
    # Visualiser la convergence
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('TensorFlow - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['mae'], label='Train MAE')
    plt.plot(history.history['val_mae'], label='Val MAE')
    plt.title('TensorFlow - MAE')
    plt.xlabel('Epochs')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    tf_success = True
    
    # Créer un wrapper pour compatibilité avec benchmark
    class TFModelWrapper:
        def __init__(self, model, scaler_x, scaler_y):
            self.model = model
            self.scaler_x = scaler_x
            self.scaler_y = scaler_y
            
        def predict(self, dataset):
            # Extraire les données
            X_test = []
            for attr in attr_x:
                if attr in dataset.data:
                    X_test.append(dataset.data[attr])
            X_test = np.column_stack(X_test)
            
            # Prédiction
            X_test_scaled = self.scaler_x.transform(X_test)
            y_pred_scaled = self.model.predict(X_test_scaled)
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled)
            
            return y_pred
    
    tf_model_wrapper = TFModelWrapper(tf_model, scaler_X, scaler_y)
    
except Exception as e:
    print(f" Erreur avec TensorFlow: {e}")
    tf_success = False
    tf_model_wrapper = None

In [ ]:
##MODÈLE BASELINE SIMPLE AVEC PYTORCH

print(f"\n CRÉATION DU MODÈLE PYTORCH BASELINE")

try:
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    
    # Convertir les données en tenseurs PyTorch
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.FloatTensor(y_train_scaled)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val_scaled)
    
    # Créer les DataLoaders
    train_dataset_torch = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset_torch = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset_torch, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_dataset_torch, batch_size=1024, shuffle=False)
    
    # Définir le modèle PyTorch
    class AirfoilNet(nn.Module):
        def __init__(self, input_size, output_size):
            super(AirfoilNet, self).__init__()
            self.network = nn.Sequential(
                nn.Linear(input_size, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_size)
            )
        
        def forward(self, x):
            return self.network(x)
    
    # Créer le modèle
    torch_model = AirfoilNet(X_train_scaled.shape[1], y_train_scaled.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)
    
    print(" Modèle PyTorch créé")
    print(f" Paramètres: {sum(p.numel() for p in torch_model.parameters())}")
    
    # Entraînement
    print(f"🏃 Entraînement en cours...")
    start_time = time.time()
    
    train_losses = []
    val_losses = []
    
    for epoch in range(20):
        # Phase d'entraînement
        torch_model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = torch_model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Phase de validation
        torch_model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                outputs = torch_model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
        
        train_losses.append(train_loss / len(train_loader))
        val_losses.append(val_loss / len(val_loader))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/20 - Train Loss: {train_losses[-1]:.6f}, Val Loss: {val_losses[-1]:.6f}")
    
    train_time_torch = time.time() - start_time
    print(f" Entraînement terminé en {train_time_torch:.1f}s")
    
    # Visualiser la convergence
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('PyTorch - Convergence')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    torch_success = True
    
    # Créer un wrapper pour compatibilité avec benchmark
    class TorchModelWrapper:
        def __init__(self, model, scaler_x, scaler_y):
            self.model = model
            self.scaler_x = scaler_x
            self.scaler_y = scaler_y
            
        def predict(self, dataset):
            # Extraire les données
            X_test = []
            for attr in attr_x:
                if attr in dataset.data:
                    X_test.append(dataset.data[attr])
            X_test = np.column_stack(X_test)
            
            # Prédiction
            X_test_scaled = self.scaler_x.transform(X_test)
            X_test_tensor = torch.FloatTensor(X_test_scaled)
            
            self.model.eval()
            with torch.no_grad():
                y_pred_scaled = self.model(X_test_tensor).numpy()
            
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled)
            return y_pred
    
    torch_model_wrapper = TorchModelWrapper(torch_model, scaler_X, scaler_y)
    
except Exception as e:
    print(f" Erreur avec PyTorch: {e}")
    torch_success = False
    torch_model_wrapper = None

In [ ]:
##ÉVALUATION DES MODÈLES

print(f"\n ÉVALUATION DES MODÈLES BASELINE")


results = {}

# Évaluation TensorFlow
if tf_success and tf_model_wrapper:
    try:
        print(f" Évaluation TensorFlow...")
        start_time = time.time()
        
        predictions_tf = tf_model_wrapper.predict(test_dataset)
        
        # Calculer des métriques simples
        y_true = []
        for attr in attr_y:
            if attr in test_dataset.data:
                y_true.append(test_dataset.data[attr])
        y_true = np.column_stack(y_true)
        
        mse_tf = np.mean((predictions_tf - y_true) ** 2)
        mae_tf = np.mean(np.abs(predictions_tf - y_true))
        
        eval_time = time.time() - start_time
        
        results["TensorFlow"] = {
            "train_time": train_time,
            "eval_time": eval_time,
            "mse": mse_tf,
            "mae": mae_tf,
            "predictions_shape": predictions_tf.shape,
            "success": True
        }
        
        print(f" TensorFlow évalué en {eval_time:.1f}s")
        print(f"   MSE: {mse_tf:.6f}")
        print(f"   MAE: {mae_tf:.6f}")
        
    except Exception as e:
        print(f" Erreur évaluation TensorFlow: {e}")
        results["TensorFlow"] = {"success": False, "error": str(e)}

# Évaluation PyTorch
if torch_success and torch_model_wrapper:
    try:
        print(f" Évaluation PyTorch...")
        start_time = time.time()
        
        predictions_torch = torch_model_wrapper.predict(test_dataset)
        
        mse_torch = np.mean((predictions_torch - y_true) ** 2)
        mae_torch = np.mean(np.abs(predictions_torch - y_true))
        
        eval_time = time.time() - start_time
        
        results["PyTorch"] = {
            "train_time": train_time_torch,
            "eval_time": eval_time,
            "mse": mse_torch,
            "mae": mae_torch,
            "predictions_shape": predictions_torch.shape,
            "success": True
        }
        
        print(f"   PyTorch évalué en {eval_time:.1f}s")
        print(f"   MSE: {mse_torch:.6f}")
        print(f"   MAE: {mae_torch:.6f}")
        
    except Exception as e:
        print(f" Erreur évaluation PyTorch: {e}")
        results["PyTorch"] = {"success": False, "error": str(e)}

In [ ]:
##ANALYSE COMPARATIVE DES RÉSULTATS

print(f"\n ANALYSE COMPARATIVE DES RÉSULTATS")


successful_models = [name for name, result in results.items() if result.get("success", False)]

if successful_models:
    print(f" Modèles réussis: {successful_models}")
    
    # Créer un graphique de comparaison
    if len(successful_models) >= 2:
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Comparaison Modèles Baseline', fontsize=16, fontweight='bold')
        
        # Temps d'entraînement
        names = successful_models
        train_times = [results[name]["train_time"] for name in names]
        
        axes[0,0].bar(names, train_times, color=['blue', 'red'], alpha=0.7)
        axes[0,0].set_ylabel('Temps (s)')
        axes[0,0].set_title('Temps d\'Entraînement')
        axes[0,0].grid(True, alpha=0.3)
        
        # MSE Comparison
        mse_values = [results[name]["mse"] for name in names]
        axes[0,1].bar(names, mse_values, color=['blue', 'red'], alpha=0.7)
        axes[0,1].set_ylabel('MSE')
        axes[0,1].set_title('Mean Squared Error')
        axes[0,1].grid(True, alpha=0.3)
        
        # MAE Comparison
        mae_values = [results[name]["mae"] for name in names]
        axes[1,0].bar(names, mae_values, color=['blue', 'red'], alpha=0.7)
        axes[1,0].set_ylabel('MAE')
        axes[1,0].set_title('Mean Absolute Error')
        axes[1,0].grid(True, alpha=0.3)
        
        # Temps d'évaluation
        eval_times = [results[name]["eval_time"] for name in names]
        axes[1,1].bar(names, eval_times, color=['blue', 'red'], alpha=0.7)
        axes[1,1].set_ylabel('Temps (s)')
        axes[1,1].set_title('Temps d\'Évaluation')
        axes[1,1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Tableau de comparaison
    print(f"\n TABLEAU COMPARATIF:")
    print(f"{'Modèle':<12} {'Train(s)':<10} {'Eval(s)':<10} {'MSE':<12} {'MAE':<12}")
    print("-" * 60)
    
    for name in successful_models:
        result = results[name]
        print(f"{name:<12} {result['train_time']:<10.1f} {result['eval_time']:<10.3f} {result['mse']:<12.6f} {result['mae']:<12.6f}")

else:
    print(f" Aucun modèle n'a réussi l'évaluation complète")

In [ ]:
## 9. CONCLUSIONS ET RECOMMANDATIONS

print(f"\n CONCLUSIONS ET RECOMMANDATIONS")
print("=" * 50)

if successful_models:
    # Trouver le meilleur modèle
    best_model = min(successful_models, key=lambda x: results[x]["mse"])
    print(f" MEILLEUR MODÈLE (MSE): {best_model}")
    print(f"   MSE: {results[best_model]['mse']:.6f}")
    print(f"   MAE: {results[best_model]['mae']:.6f}")
    print(f"   Temps d'entraînement: {results[best_model]['train_time']:.1f}s")
    
    print(f"\n RECOMMANDATIONS POUR L'AMÉLIORATION:")
    print(f"   • Augmenter le nombre d'époques (50-100)")
    print(f"   • Essayer différentes architectures (plus de couches)")
    print(f"   • Optimiser les hyperparamètres (learning rate, batch size)")
    print(f"   • Ajouter de la régularisation (L1, L2)")
    print(f"   • Utiliser des techniques avancées (attention, residual connections)")
    
    print(f"\n PROCHAINES ÉTAPES:")
    print(f"   • Notebook 3: Optimisation des hyperparamètres")
    print(f"   • Notebook 4: Architectures avancées")
    print(f"   • Notebook 5: Évaluation physique complète")

print(f"\n BASELINE TERMINÉ - Modèles de référence établis!")

# Sauvegarder les résultats
import pickle
with open('baseline_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print(f" Résultats sauvegardés dans 'baseline_results.pkl'")